In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    db_path='faq.db'
)

In [2]:
sqlite_index.count()

470

In [3]:
from dotenv import load_dotenv
import os
load_dotenv()

from openai import OpenAI
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [4]:
from rag_helper import RAGBase

assistant = RAGBase(sqlite_index, openai_client)

In [5]:
assistant.rag('How do I submit homework?')

"I don't know."

In [21]:
assistant.rag('Are Jupyter Notebooks used?')

"I don't know."

In [16]:
sqlite_index.count()

470

In [11]:
# Search in question
results = sqlite_index.search("Are Jupyter Notebooks used?", num_results=5)
[doc["question"] for doc in results]

['Are Jupyter Notebooks used?',
 'How to Disable/avoid Warnings in Jupyter Notebooks',
 'Setting up an environment using VS Code',
 'Downloading a csv file inside notebook',
 'Loading the dataset directly through Kaggle Notebooks']

In [9]:
results = sqlite_index.search("Are Jupyter Notebooks used?", num_results=5)
[doc["section"] for doc in results]

['General Course-Related Questions',
 'Module 3. Machine Learning for Classification',
 'Module 1. Introduction to Machine Learning',
 'Module 1. Introduction to Machine Learning',
 'Module 2. Machine Learning for Regression']

In [8]:
results

[{'course': 'machine-learning',
  'section': 'General Course-Related Questions',
  'question': 'Are Jupyter Notebooks used?',
  'answer': 'Yes. You’ll work extensively with notebooks alongside standard Python files and CLI tools.',
  'doc_id': '4d5aa45b03'},
 {'course': 'machine-learning',
  'section': 'Module 3. Machine Learning for Classification',
  'question': 'How to Disable/avoid Warnings in Jupyter Notebooks',
  'answer': 'The warnings in Jupyter notebooks can be disabled or avoided with the following commands:\n\n```python\nimport warnings\n\nwarnings.filterwarnings("ignore")\n```',
  'doc_id': '9610a0de1d'},
 {'course': 'machine-learning',
  'section': 'Module 1. Introduction to Machine Learning',
  'question': 'Setting up an environment using VS Code',
  'answer': 'I found this video quite helpful: [Creating Virtual Environment for Python from VS Code](https://www.youtube.com/watch?v=8h9w0meM8i4)\n\n**Native Jupyter Notebooks Support in VS Code**\n\nIn VS Code, you can have n

In [12]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

custom_instructions

"You're a course teaching assistant.\nAnswer the QUESTION based on the CONTEXT from the FAQ database.\nUse only the facts from the CONTEXT when answering the QUESTION."

In [ ]:
assistant_custom = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [16]:
new_prompt = assistant_custom.build_prompt("Are Jupyter Notebooks used?", results)
new_prompt

General Course-Related Questions
Q: Are Jupyter Notebooks used?
A: Yes. You’ll work extensively with notebooks alongside standard Python files and CLI tools.

Module 3. Machine Learning for Classification
Q: How to Disable/avoid Warnings in Jupyter Notebooks
A: The warnings in Jupyter notebooks can be disabled or avoided with the following commands:

```python
import warnings

warnings.filterwarnings("ignore")
```

Module 1. Introduction to Machine Learning
Q: Setting up an environment using VS Code
A: I found this video quite helpful: [Creating Virtual Environment for Python from VS Code](https://www.youtube.com/watch?v=8h9w0meM8i4)

**Native Jupyter Notebooks Support in VS Code**

In VS Code, you can have native Jupyter Notebooks support, i.e., you do not need to open a web browser to code in a Notebook. If you have port forwarding enabled, run a `jupyter notebook` command from a remote machine, and have a remote connection configured in `.ssh/config` (as Alexey’s [video](https://www

'QUESTION: Are Jupyter Notebooks used?\n\nCONTEXT:\nGeneral Course-Related Questions\nQ: Are Jupyter Notebooks used?\nA: Yes. You’ll work extensively with notebooks alongside standard Python files and CLI tools.\n\nModule 3. Machine Learning for Classification\nQ: How to Disable/avoid Warnings in Jupyter Notebooks\nA: The warnings in Jupyter notebooks can be disabled or avoided with the following commands:\n\n```python\nimport warnings\n\nwarnings.filterwarnings("ignore")\n```\n\nModule 1. Introduction to Machine Learning\nQ: Setting up an environment using VS Code\nA: I found this video quite helpful: [Creating Virtual Environment for Python from VS Code](https://www.youtube.com/watch?v=8h9w0meM8i4)\n\n**Native Jupyter Notebooks Support in VS Code**\n\nIn VS Code, you can have native Jupyter Notebooks support, i.e., you do not need to open a web browser to code in a Notebook. If you have port forwarding enabled, run a `jupyter notebook` command from a remote machine, and have a remote

In [17]:
answer = assistant_custom.llm(new_prompt)
answer

'Yes.\u202fYou’ll work extensively with Jupyter Notebooks alongside standard Python files and CLI tools.'

In [18]:
assistant_custom.rag("Are Jupyter Notebooks used?")

'Based on the provided context, there is no information about whether Jupyter Notebooks are used.'

In [26]:
assistant_custom.rag("How do I submit homework?")

'I’m sorry, but I don’t have the information needed to answer your question.'

In [51]:

boost_dict = {'question': 3.0, 'section': 0.5}
filter_dict = {'course': "machine-learning"}

search_results = sqlite_index.search(
    "Are Jupyter Notebooks used?",
    num_results=3,
    boost_dict=boost_dict,
    filter_dict=filter_dict)

search_results

[{'course': 'machine-learning',
  'section': 'General Course-Related Questions',
  'question': 'Are Jupyter Notebooks used?',
  'answer': 'Yes. You’ll work extensively with notebooks alongside standard Python files and CLI tools.',
  'doc_id': '4d5aa45b03'},
 {'course': 'machine-learning',
  'section': 'Module 3. Machine Learning for Classification',
  'question': 'How to Disable/avoid Warnings in Jupyter Notebooks',
  'answer': 'The warnings in Jupyter notebooks can be disabled or avoided with the following commands:\n\n```python\nimport warnings\n\nwarnings.filterwarnings("ignore")\n```',
  'doc_id': '9610a0de1d'},
 {'course': 'machine-learning',
  'section': 'Module 1. Introduction to Machine Learning',
  'question': 'Setting up an environment using VS Code',
  'answer': 'I found this video quite helpful: [Creating Virtual Environment for Python from VS Code](https://www.youtube.com/watch?v=8h9w0meM8i4)\n\n**Native Jupyter Notebooks Support in VS Code**\n\nIn VS Code, you can have n

In [53]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')
    
    return '\n'.join(lines).strip()

In [56]:
context = build_context(search_results)
context.strip()

'General Course-Related Questions\nQ: Are Jupyter Notebooks used?\nA: Yes. You’ll work extensively with notebooks alongside standard Python files and CLI tools.\n\nModule 3. Machine Learning for Classification\nQ: How to Disable/avoid Warnings in Jupyter Notebooks\nA: The warnings in Jupyter notebooks can be disabled or avoided with the following commands:\n\n```python\nimport warnings\n\nwarnings.filterwarnings("ignore")\n```\n\nModule 1. Introduction to Machine Learning\nQ: Setting up an environment using VS Code\nA: I found this video quite helpful: [Creating Virtual Environment for Python from VS Code](https://www.youtube.com/watch?v=8h9w0meM8i4)\n\n**Native Jupyter Notebooks Support in VS Code**\n\nIn VS Code, you can have native Jupyter Notebooks support, i.e., you do not need to open a web browser to code in a Notebook. If you have port forwarding enabled, run a `jupyter notebook` command from a remote machine, and have a remote connection configured in `.ssh/config` (as Alexey’